# Final Test Results

Runs test set with optuna tuned HPs for all 21 **model x representation** combinations. These are the results reported in Table 1.

In [2]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path('/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/PBSG')))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
df = pd.read_csv('data_classifier.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [5]:
with open("random_stratified_splits_cls.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=546  test=97  folds=5
Device: cpu


In [7]:
X_fp_RU        = np.load("Vectors/X_fp_RU.npy")
X_fp_pooled_RU    = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly      = np.load("Vectors/X_fp_poly.npy")
X_fp_pooled_poly = np.load("Vectors/X_fp_pooled_poly.npy")

graphs_PBSG = torch.load("Graphs/graphs_PBSG.pt")
graphs_SMILES = torch.load("Graphs/graphs_SMILES.pt")
graphs_SMILESplusglobal = torch.load("Graphs/graphs_SMILES+global.pt")

y=np.load("Vectors/y.npy")

In [10]:
# Inject y into graphs first
graph_reps = {
    "PBSG":          graphs_PBSG,
    "SMILES":        graphs_SMILES,
    "SMILES+global": graphs_SMILESplusglobal
}

for graphs in graph_reps.values():
    for i, g in enumerate(graphs):
        if g.y is None:
            import torch
            g.y = torch.tensor([y[i]], dtype=torch.float32)

In [12]:
from scripts.test_eval_all_models_cls import run_all_test

test_results = run_all_test(
    graphs_PBSG, graphs_SMILES, graphs_SMILESplusglobal,
    X_fp_pooled_RU, X_fp_RU, X_fp_pooled_poly, X_fp_poly,
    y, df, trainval_idx, test_idx, device,
    hp_path="best_hp_full.json",
)

Loaded HPs from best_hp_full.json  (21 entries)

  Retraining GCN (PBSG) on full trainval (n=546)
  hidden=256, layers=3, dropout=0.295, lr=0.00386, seed=0
    Epoch  20 | train_loss=0.3256 val_loss=0.5335 val_auroc=0.8588 val_auprc=0.9216
    Early stop at epoch 32  best_auroc=0.8948
  Training time: 0.18 min

  TEST RESULTS: GCN (PBSG)
  Accuracy              : 0.8247
  Macro F1              : 0.7930
  AUROC                 : 0.8948
  AUPRC                 : 0.9348
  No Tm Precision       : 0.9545
  No Tm Recall          : 0.5676
  Has Tm Precision      : 0.7867
  Has Tm Recall         : 0.9833
  train_time_min        : 0.1800

  Subgroup: copolymer_type
  Group                    n      Acc       F1
  ---------------------------------------------
  alternating              9   0.5556   0.7143
  block                    2   1.0000   1.0000
  random                  30   0.8667   0.9286

  Subgroup: stereo_class
  Group                    n      Acc       F1
  ------------------------

/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



  TEST RESULTS: LR (FP+pooled RU)
  Accuracy              : 0.8660
  Macro F1              : 0.8612
  AUROC                 : 0.9151
  AUPRC                 : 0.9457
  No Tm Precision       : 0.7857
  No Tm Recall          : 0.8919
  Has Tm Precision      : 0.9273
  Has Tm Recall         : 0.8500
  train_time_min        : 0.0400

  Subgroup: copolymer_type
  Group                    n      Acc       F1
  ---------------------------------------------
  alternating              9   0.5556   0.6000
  block                    2   1.0000   1.0000
  random                  30   0.9333   0.9600

  Subgroup: stereo_class
  Group                    n      Acc       F1
  ---------------------------------------------
  achiral                  5   0.6000   0.6667
  atactic                 19   0.8947   0.8571
  stereo-irregular             9   0.8889   0.6667
  isotactic               45   0.8889   0.9333
  syndiotactic            19   0.8421   0.8235

── LR (FP RU) ─────────────────────────────

In [15]:
import json
with open("saved_models_all/cls_thresholds-full.json") as f:
    print(json.load(f))
# Look for "xgb_fp_pooled_poly" key

{'gcn_pbsg': 0.342664510011673, 'gcn_smiles': 0.6455289721488953, 'gcn_smiles_gl': 0.5560771226882935, 'gine_pbsg': 0.40857556462287903, 'gine_smiles': 0.461488276720047, 'gine_smiles_gl': 0.34616944193840027, 'gatv2_pbsg': 0.43857866525650024, 'gatv2_smiles': 0.27774718403816223, 'gatv2_smiles_gl': 0.3173174262046814, 'lr_fp_pooled_ru': 0.6416157209602327, 'lr_fp_ru': 0.38534373789692355, 'lr_fp_pooled_poly': 0.24772134301473323, 'lr_fp_poly': 0.21664432535450365, 'rf_fp_pooled_ru': 0.5314108725492189, 'rf_fp_ru': 0.4552430511233865, 'rf_fp_pooled_poly': 0.600665202391843, 'rf_fp_poly': 0.34794008144742616, 'xgb_fp_pooled_ru': 0.6187085509300232, 'xgb_fp_ru': 0.49966394901275635, 'xgb_fp_pooled_poly': 0.8112525939941406, 'xgb_fp_poly': 0.4311915338039398}


In [21]:
%run scripts/train_final_full_data_cls.py

Full classifier data: N=643  pos=407 neg=236
  saved -> saved_models_final_fulldata/cls_xgb_pooled_poly_FULL.pkl
  carried over threshold xgb_fp_pooled_poly=0.8112525939941406 (from reported held-out run)

Done -> saved_models_final_fulldata/  (EXTERNAL predictions only)
